# 🚀 QuantumFlow AI Trading System v2.0
## Analysis Notebook

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from features.engineering import AdvancedFeatureEngineer
from env.trading_env import QuantumTradingEnv
from agents.policy_network import TransformerPolicyNetwork, EnsemblePolicy
from evaluation.backtest import BacktestEngine

## 1. Load Data

In [ ]:
# Load your data here
# df = pd.read_csv('data/XAUUSD_M5.csv')

## 2. Feature Engineering

In [ ]:
engineer = AdvancedFeatureEngineer(config={})
features = engineer.compute_all_features(df)
print(f'Features shape: {features.shape}')

## 3. Visualize Features

In [ ]:
plt.figure(figsize=(15, 10))
for i in range(min(9, features.shape[1])):
    plt.subplot(3, 3, i+1)
    plt.plot(features[:, i])
    plt.title(f'Feature {i}')
plt.tight_layout()
plt.show()

## 4. Create Environment

In [ ]:
returns = df['close'].pct_change().fillna(0).values
timestamps = np.arange(len(df))

env = QuantumTradingEnv(features, returns, timestamps, window=128)

## 5. Load Trained Agent

In [ ]:
agent = TransformerPolicyNetwork(n_features=features.shape[1], window_size=128)
# agent.load_state_dict(torch.load('checkpoints/best_model.pt')['policy_state_dict'])

## 6. Backtest

In [ ]:
backtest = BacktestEngine(config={'initial_capital': 100000})
result = backtest.run_backtest(agent, env)

print(f'Total Return: {result.total_return:.2%}')
print(f'Sharpe Ratio: {result.sharpe_ratio:.2f}')
print(f'Max Drawdown: {result.max_drawdown:.2%}')

## 7. Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Equity curve
axes[0].plot(result.equity_curve)
axes[0].set_title('Equity Curve')
axes[0].set_ylabel('Equity ($)')

# Drawdown
axes[1].fill_between(range(len(result.drawdown_curve)), result.drawdown_curve * 100, 0, color='red', alpha=0.3)
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Time')

plt.tight_layout()
plt.show()